# Ocean Proximity als Feature

In diesem Notebook wird die Idee untersucht, geografische Cluster als zusätzliches Feature einzuführen und mit einer One-Hot-Encoding-Darstellung in das Modell einzuarbeiten.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow.keras as keras
from sklearn.datasets import fetch_california_housing

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

2026-03-26 14:32:14.654871: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-26 14:32:14.662303: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-26 14:32:14.955377: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-26 14:32:16.104234: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

In [ ]:
SEED = 42
SF_COORDS = (37.7749, -122.4194)
LA_COORDS = (34.0522, -118.2437)
SJ_COORDS = (37.3362, -121.8833)


def prepare_dataset():
    data = fetch_california_housing()
    df = pd.DataFrame(data=data.data, columns=data.feature_names)
    df[data.target_names[0]] = data.target

    mask_cutoff = (
        (df["MedHouseVal"] < df["MedHouseVal"].max())
        & (df["HouseAge"] < df["HouseAge"].max())
        & (df["MedInc"] < df["MedInc"].max())
    )
    df_clean = df[mask_cutoff].copy()

    clip_cols = ["AveRooms", "AveBedrms", "AveOccup", "Population"]
    for col in clip_cols:
        lower_bound = df_clean[col].quantile(0.01)
        upper_bound = df_clean[col].quantile(0.99)
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)

    df_clean["BedrmsPerRoom"] = df_clean["AveBedrms"] / df_clean["AveRooms"]

    for col in ["MedInc", "Population", "AveBedrms"]:
        df_clean[col] = np.log1p(df_clean[col])

    df_clean["Dist_to_SF"] = np.sqrt(
        (df_clean["Latitude"] - SF_COORDS[0]) ** 2 + (df_clean["Longitude"] - SF_COORDS[1]) ** 2
    )
    df_clean["Dist_to_LA"] = np.sqrt(
        (df_clean["Latitude"] - LA_COORDS[0]) ** 2 + (df_clean["Longitude"] - LA_COORDS[1]) ** 2
    )
    df_clean["Dist_to_SJ"] = np.sqrt(
        (df_clean["Latitude"] - SJ_COORDS[0]) ** 2 + (df_clean["Longitude"] - SJ_COORDS[1]) ** 2
    )

    X = df_clean.drop("MedHouseVal", axis=1)
    y = df_clean["MedHouseVal"].copy()

    X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )

    scaler = sklearn.preprocessing.StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return df_clean, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled


df_clean, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled = prepare_dataset()

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="linear"),
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=200,
    batch_size=64,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    ],
    verbose=0,
)

test_scores = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"MSE: {test_scores[0]:.2f}")
print(f"MAE: {test_scores[1]:.2f}")

Epoch 1/200


2026-03-26 14:32:17.373141: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.0260 - mae: 1.3844 - val_loss: 1.3057 - val_mae: 0.8594 - learning_rate: 0.0010
Epoch 2/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.1176 - mae: 0.7662 - val_loss: 0.4868 - val_mae: 0.4448 - learning_rate: 0.0010
Epoch 3/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7954 - mae: 0.6348 - val_loss: 0.3986 - val_mae: 0.3830 - learning_rate: 0.0010
Epoch 4/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7018 - mae: 0.5827 - val_loss: 0.3873 - val_mae: 0.3800 - learning_rate: 0.0010
Epoch 5/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6210 - mae: 0.5411 - val_loss: 0.3625 - val_mae: 0.3609 - learning_rate: 0.0010
Epoch 6/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5704 - mae: 0.5094 - val_loss: 0.3574 - val_mae: 0.3589 - learning_rate: 0.0010
Epoch 7/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5236 - mae: 0.4870 - val_loss: 0.3547 - val_mae: 0.3556 - learning_rate: 0.0010
Epoch 8/200

## Motivation

Da Kalifornien eine sehr beliebte Küste hat und viele teure Regionen an der Küste liegen, ist es naheliegend, ein geografisches Feature zu erzeugen, das die Lage der Datenpunkte beschreibt.

- Idee: K-Means-Clustering auf den Koordinaten anwenden und jeden Datenpunkt einer Cluster-Kategorie zuordnen.
- Danach wird das Cluster-Feature mit One-Hot-Encoding verarbeitet, damit unterschiedliche Cluster nicht implizit unterschiedlich gewichtet werden.
- Wichtig: Das Clustering sollte auf den unskalierten Koordinaten erfolgen, damit die Datenstruktur nicht durch Standardisierung verzerrt wird.

In [5]:
from sklearn.cluster import KMeans

In [ ]:
df_clean.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal', 'BedrmsPerRoom', 'Dist_to_SF',
       'Dist_to_LA', 'Dist_to_SJ'],
      dtype='str')

In [ ]:
from sklearn.cluster import KMeans

SEED = 42
SF_COORDS = (37.7749, -122.4194)
LA_COORDS = (34.0522, -118.2437)
SJ_COORDS = (37.3362, -121.8833)


data = fetch_california_housing()
df = pd.DataFrame(data=data.data, columns=data.feature_names)
df["MedHouseVal"] = data.target

mask_cutoff = (
    (df["MedHouseVal"] < df["MedHouseVal"].max())
    & (df["HouseAge"] < df["HouseAge"].max())
    & (df["MedInc"] < df["MedInc"].max())
)
df_clean = df[mask_cutoff].copy()

clip_cols = ["AveRooms", "AveBedrms", "AveOccup", "Population"]
for col in clip_cols:
    df_clean[col] = df_clean[col].clip(
        lower=df_clean[col].quantile(0.01),
        upper=df_clean[col].quantile(0.99),
    )

df_clean["BedrmsPerRoom"] = df_clean["AveBedrms"] / df_clean["AveRooms"]

for col in ["MedInc", "Population", "AveBedrms"]:
    df_clean[col] = np.log1p(df_clean[col])

df_clean["Dist_to_SF"] = np.sqrt(
    (df_clean["Latitude"] - SF_COORDS[0]) ** 2 + (df_clean["Longitude"] - SF_COORDS[1]) ** 2
)
df_clean["Dist_to_LA"] = np.sqrt(
    (df_clean["Latitude"] - LA_COORDS[0]) ** 2 + (df_clean["Longitude"] - LA_COORDS[1]) ** 2
)
df_clean["Dist_to_SJ"] = np.sqrt(
    (df_clean["Latitude"] - SJ_COORDS[0]) ** 2 + (df_clean["Longitude"] - SJ_COORDS[1]) ** 2
)

X = df_clean.drop("MedHouseVal", axis=1)
y = df_clean["MedHouseVal"].copy()

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

coords_train = X_train[["Latitude", "Longitude"]]
coords_test = X_test[["Latitude", "Longitude"]]

kmeans = KMeans(n_clusters=10, n_init=10, random_state=SEED)
train_clusters = kmeans.fit_predict(coords_train)
test_clusters = kmeans.predict(coords_test)

oh_encoder = sklearn.preprocessing.OneHotEncoder(sparse_output=False)
train_clusters_oh = oh_encoder.fit_transform(train_clusters.reshape(-1, 1))
test_clusters_oh = oh_encoder.transform(test_clusters.reshape(-1, 1))

scaler = sklearn.preprocessing.StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_final = np.hstack([X_train_scaled, train_clusters_oh])
X_test_final = np.hstack([X_test_scaled, test_clusters_oh])

print(f"Start Training. Anzahl der finalen Features: {X_train_final.shape[1]}")

Start Training. Anzahl der finalen Features: 22


In [ ]:
model_op = keras.Sequential([
    keras.layers.Input(shape=(X_train_final.shape[1],)),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="linear"),
])

model_op.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model_op.fit(
    X_train_final,
    y_train,
    epochs=200,
    batch_size=64,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    ],
    verbose=0,
)

test_scores_op = model_op.evaluate(X_test_final, y_test, verbose=0)
print(f"MSE: {test_scores_op[0]:.2f}")
print(f"MAE: {test_scores_op[1]:.2f}")

Epoch 1/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3.0439 - mae: 1.3984 - val_loss: 1.2936 - val_mae: 0.8578 - learning_rate: 0.0010
Epoch 2/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0840 - mae: 0.7532 - val_loss: 0.4491 - val_mae: 0.3992 - learning_rate: 0.0010
Epoch 3/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7758 - mae: 0.6202 - val_loss: 0.3932 - val_mae: 0.3670 - learning_rate: 0.0010
Epoch 4/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6706 - mae: 0.5624 - val_loss: 0.3682 - val_mae: 0.3508 - learning_rate: 0.0010
Epoch 5/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5948 - mae: 0.5216 - val_loss: 0.3593 - val_mae: 0.3452 - learning_rate: 0.0010
Epoch 6/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5483 - mae: 0.4935 - val_loss: 0.3509 - val_mae: 0.3449 - learning_rate: 0.0010
Epoch 7/200
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5126 - mae: 0.4759 - val_loss: 0.3456 - val_mae: 0.3395 - learning_rate: 0.0010

In [ ]:
print(
    f"--- Test-Scores ---\n"
    f"Ohne Ocean-Proximity: MAE: {test_scores[0]:.2f}, MSE: {test_scores[1]:.2f}\n"
    f"Mit Ocean-Proximity: MAE: {test_scores_op[0]:.2f}, MSE: {test_scores_op[1]:.2f}"
)

--- Test-Scores ---
Ohne Ocean-Proximity: MAE: 0.22, MSE: 0.30
Mit Ocean-Proximity: MAE. 0.22, MSE: 0.31


In [ ]:
from sklearn.metrics import r2_score

y_pred = model_op.predict(X_test_final)
r2 = r2_score(y_test, y_pred)

print(f"R² Score auf den Testdaten: {r2:.4f}")

117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
R² Score auf den Testdaten: 0.7841


In [ ]:
# Abschlusszelle für die Analyse.